<a href="https://colab.research.google.com/github/rayan4534/NLP-project/blob/main/Projet_ASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install datasets transformers accelerate evaluate jiwer sentencepiece soundfile sacremoses

import pandas as pd, torch, re, os, unicodedata
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ {device} | GPU: {torch.cuda.get_device_name(0) if device=='cuda' else 'N/A'}")

In [ ]:
ds_map = {
    "Tamazight-NLP/tamawalt-n-imZZyann": load_dataset("Tamazight-NLP/tamawalt-n-imZZyann"),
    "Tamazight-NLP/Tamazight-Speech-to-Arabic-Text": load_dataset("Tamazight-NLP/Tamazight-Speech-to-Arabic-Text"),
    "TutlaytAI/Kabyle_ASR-Fr_Translation": load_dataset("TutlaytAI/Kabyle_ASR-Fr_Translation"),
    "Tamazight-NLP/Beni-Mellal-Tamazight": load_dataset("Tamazight-NLP/Beni-Mellal-Tamazight"),
    "Tamazight-NLP/TOSD": load_dataset("Tamazight-NLP/TOSD"),
}
# SoufianeDahimi/Tamazight-ASR-Dataset-v2 exclu (doublon exact de Speech-to-Arabic)

for name, ds in ds_map.items():
    print(f"  {name}: {sum(len(ds[s]) for s in ds)} lignes")

In [ ]:
# === Conversions ===
DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
ARABIZI = {"ا":"a","أ":"a","إ":"i","آ":"a","ء":"'","ؤ":"w","ئ":"y","ى":"a",
    "ب":"b","ت":"t","ث":"th","ج":"j","ح":"7","خ":"5","د":"d","ذ":"dh",
    "ر":"r","ز":"z","س":"s","ش":"sh","ص":"s","ض":"d","ط":"t","ظ":"z",
    "ع":"3","غ":"8","ف":"f","ق":"9","ك":"k","ل":"l","م":"m","ن":"n",
    "ه":"h","ة":"a","و":"w","ي":"y","ـ":""}
TIF = {"ⴰ":"a","ⴱ":"b","ⴳ":"g","ⴷ":"d","ⴹ":"ḍ","ⴻ":"e","ⴼ":"f","ⴽ":"k",
    "ⵀ":"h","ⵃ":"ḥ","ⵄ":"ɛ","ⵅ":"x","ⵇ":"q","ⵉ":"i","ⵊ":"j","ⵍ":"l",
    "ⵎ":"m","ⵏ":"n","ⵓ":"u","ⵔ":"r","ⵕ":"ṛ","ⵖ":"ɣ","ⵙ":"s","ⵚ":"ṣ",
    "ⵛ":"c","ⵜ":"t","ⵟ":"ṭ","ⵡ":"w","ⵢ":"y","ⵣ":"z","ⵥ":"ẓ"}
TIF_RE = re.compile(r"[\u2D30-\u2D7F]")

def ar2abz(t):
    if pd.isna(t) or str(t).strip()=="": return ""
    t = DIACRITICS.sub("", str(t))
    return re.sub(r"\s+"," ","".join(ARABIZI.get(c,c) for c in t)).strip().lower()

def lat2abz(t):
    if pd.isna(t) or str(t).strip()=="": return ""
    s = unicodedata.normalize("NFKC", str(t).strip().lower())
    if TIF_RE.search(s):
        s = "".join(TIF.get(c,c) for c in s)
    s = (s.replace("ḥ","7").replace("ɛ","3").replace("ɣ","8").replace("x","5")
          .replace("q","9").replace("ḍ","d").replace("ṭ","t").replace("ṣ","s")
          .replace("ẓ","z").replace("ṛ","r").replace("č","ch").replace("c","ch"))
    return re.sub(r"\s+"," ",s).strip()

def audio_path(x):
    return x.get("path","") if isinstance(x,dict) else ("" if x is None else str(x))

def cl(x):
    return "" if x is None or (isinstance(x,float) and pd.isna(x)) else str(x).strip()

# === Config par dataset ===
CFG = {
    "Tamazight-NLP/tamawalt-n-imZZyann":
        {"audio":"audio","lat":"Tamazight_word","ar":"Arabic_meaning","en":"English_meaning","fr":"French_meaning","conv":lat2abz},
    "Tamazight-NLP/Tamazight-Speech-to-Arabic-Text":
        {"audio":"audio","lat":None,"ar":"text","en":None,"fr":None,"conv":None},
    "TutlaytAI/Kabyle_ASR-Fr_Translation":
        {"audio":"audio","lat":"Kab Sentence","ar":None,"en":None,"fr":"Translated Sentence","conv":lat2abz},
    "Tamazight-NLP/Beni-Mellal-Tamazight":
        {"audio":"Audio","lat":"Tamazight","ar":None,"en":"English","fr":None,"conv":lat2abz},
    "Tamazight-NLP/TOSD":
        {"audio":"audio","lat":"transcription","ar":None,"en":None,"fr":None,"conv":lat2abz},
}

# === Compilation ===
rows = []
for ds_id, ds_dict in ds_map.items():
    c = CFG[ds_id]
    for split, data in ds_dict.items():
        for item in data:
            audio = audio_path(item.get(c["audio"],""))
            lat_raw = cl(item.get(c["lat"],"")) if c["lat"] else ""
            ar = cl(item.get(c["ar"],"")) if c["ar"] else ""
            en = cl(item.get(c["en"],"")) if c["en"] else ""
            fr = cl(item.get(c["fr"],"")) if c["fr"] else ""

            if c["conv"] and lat_raw:
                lat = c["conv"](lat_raw)
            elif not lat_raw and ar:
                lat = ar2abz(ar)
            else:
                lat = lat_raw

            rows.append({"Audio":audio,"Transcription latine":lat,
                "Transcription arabe":ar,"Traduction anglais":en,
                "Traduction française":fr,"source_dataset":ds_id})
    print(f"  ✅ {ds_id}")

df = pd.DataFrame(rows)
print(f"\n✅ {len(df)} lignes compilées")

# Diagnostic
for col in ["Transcription latine","Transcription arabe","Traduction anglais","Traduction française"]:
    v = (df[col]=="").sum()
    print(f"  {col}: {v} vides ({round((1-v/len(df))*100)}% rempli)")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
mdl = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
if device=="cuda": mdl = mdl.half()
mdl = mdl.to(device).eval()

BS = 32 if device=="cuda" else 8

def tr_batch(texts, src, tgt):
    res = []
    for i in range(0, len(texts), BS):
        b = texts[i:i+BS]
        tok.src_lang = src
        inp = tok(b, return_tensors="pt", truncation=True, max_length=256, padding=True).to(device)
        with torch.no_grad():
            out = mdl.generate(**inp, forced_bos_token_id=tok.convert_tokens_to_ids(tgt), max_new_tokens=256)
        res.extend(tok.batch_decode(out, skip_special_tokens=True))
        if (i//BS) % 100 == 0: print(f"  {i+len(b)}/{len(texts)}")
    return res

# Passe 1 : arabe -> arabizi (local, instantané)
m = (df["Transcription latine"]=="") & (df["Transcription arabe"]!="")
df.loc[m,"Transcription latine"] = df.loc[m,"Transcription arabe"].apply(ar2abz)
print(f"✅ Passe 1 (ar→arabizi) : {m.sum()} remplis")

# Passe 2 : latin -> arabe
m = (df["Transcription arabe"]=="") & (df["Transcription latine"]!="")
if m.sum()>0:
    df.loc[m,"Transcription arabe"] = tr_batch(df.loc[m,"Transcription latine"].tolist(), "kab_Latn", "arb_Arab")
print(f"✅ Passe 2 (lat→ar) : {m.sum()} remplis")

# Passe 3 : arabe -> français
m = (df["Traduction française"]=="") & (df["Transcription arabe"]!="")
if m.sum()>0:
    df.loc[m,"Traduction française"] = tr_batch(df.loc[m,"Transcription arabe"].tolist(), "arb_Arab", "fra_Latn")
print(f"✅ Passe 3 (ar→fr) : {m.sum()} remplis")

# Passe 4 : français -> anglais
m = (df["Traduction anglais"]=="") & (df["Traduction française"]!="")
if m.sum()>0:
    df.loc[m,"Traduction anglais"] = tr_batch(df.loc[m,"Traduction française"].tolist(), "fra_Latn", "eng_Latn")
print(f"✅ Passe 4 (fr→en) : {m.sum()} remplis")

# Diagnostic final
print("\n=== État final ===")
for col in ["Transcription latine","Transcription arabe","Traduction anglais","Traduction française"]:
    v = (df[col]=="").sum()
    print(f"  {col}: {v} vides ({round((1-v/len(df))*100)}% rempli)")

In [ ]:
from sklearn.model_selection import train_test_split

# Sauvegarde du dataset complet
df.to_csv("/content/dataset_final.csv", index=False)
print(f"✅ dataset_final.csv : {len(df)} lignes")

# Split 80/10/10
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/val.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

print(f"✅ train.csv : {len(train_df)} lignes ({round(len(train_df)/len(df)*100)}%)")
print(f"✅ val.csv   : {len(val_df)} lignes ({round(len(val_df)/len(df)*100)}%)")
print(f"✅ test.csv  : {len(test_df)} lignes ({round(len(test_df)/len(df)*100)}%)")

In [ ]:
# Aperçu des 3 fichiers
for name, d in [("TRAIN", train_df), ("VAL", val_df), ("TEST", test_df)]:
    print(f"\n--- {name} ({len(d)} lignes) ---")
    print(d[["Audio","Transcription latine","Transcription arabe"]].head(3).to_string())

In [ ]:
# ============================================================
# 🎯 ÉTAPE 3 : Baseline — Préparer les échantillons audio
# ============================================================
import random, gc, numpy as np
import pandas as pd, torch, re, unicodedata
from datasets import load_dataset, Audio

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_SIZE = 200
random.seed(42)

DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
ARABIZI = {"ا":"a","أ":"a","إ":"i","آ":"a","ء":"'","ؤ":"w","ئ":"y","ى":"a",
    "ب":"b","ت":"t","ث":"th","ج":"j","ح":"7","خ":"5","د":"d","ذ":"dh",
    "ر":"r","ز":"z","س":"s","ش":"sh","ص":"s","ض":"d","ط":"t","ظ":"z",
    "ع":"3","غ":"8","ف":"f","ق":"9","ك":"k","ل":"l","م":"m","ن":"n",
    "ه":"h","ة":"a","و":"w","ي":"y","ـ":""}
TIF = {"ⴰ":"a","ⴱ":"b","ⴳ":"g","ⴷ":"d","ⴹ":"ḍ","ⴻ":"e","ⴼ":"f","ⴽ":"k",
    "ⵀ":"h","ⵃ":"ḥ","ⵄ":"ɛ","ⵅ":"x","ⵇ":"q","ⵉ":"i","ⵊ":"j","ⵍ":"l",
    "ⵎ":"m","ⵏ":"n","ⵓ":"u","ⵔ":"r","ⵕ":"ṛ","ⵖ":"ɣ","ⵙ":"s","ⵚ":"ṣ",
    "ⵛ":"c","ⵜ":"t","ⵟ":"ṭ","ⵡ":"w","ⵢ":"y","ⵣ":"z","ⵥ":"ẓ"}
TIF_RE = re.compile(r"[\u2D30-\u2D7F]")

def ar2abz(t):
    if pd.isna(t) or str(t).strip()=="": return ""
    t = DIACRITICS.sub("", str(t))
    return re.sub(r"\s+"," ","".join(ARABIZI.get(c,c) for c in t)).strip().lower()

def lat2abz(t):
    if pd.isna(t) or str(t).strip()=="": return ""
    s = unicodedata.normalize("NFKC", str(t).strip().lower())
    if TIF_RE.search(s):
        s = "".join(TIF.get(c,c) for c in s)
    s = (s.replace("ḥ","7").replace("ɛ","3").replace("ɣ","8").replace("x","5")
          .replace("q","9").replace("ḍ","d").replace("ṭ","t").replace("ṣ","s")
          .replace("ẓ","z").replace("ṛ","r").replace("č","ch").replace("c","ch"))
    return re.sub(r"\s+"," ",s).strip()

print("⏭️ tamawalt skippé (URLs audio inaccessibles)")

val_samples = []
DATASETS = {
    "Tamazight-NLP/Tamazight-Speech-to-Arabic-Text": {"audio":"audio", "text":"text", "conv":ar2abz},
    "TutlaytAI/Kabyle_ASR-Fr_Translation": {"audio":"audio", "text":"Kab Sentence", "conv":lat2abz},
    "Tamazight-NLP/Beni-Mellal-Tamazight": {"audio":"Audio", "text":"Tamazight", "conv":lat2abz},
    "Tamazight-NLP/TOSD": {"audio":"audio", "text":"text", "conv":lat2abz},
}

for ds_id, cfg in DATASETS.items():
    print(f"🔄 {ds_id}...")
    ds = load_dataset(ds_id)

    for split_name, split_ds in ds.items():
        split_ds = split_ds.cast_column(cfg["audio"], Audio(sampling_rate=16000))
        indices = list(range(len(split_ds)))
        random.shuffle(indices)

        for idx in indices[:50]:
            item = split_ds[idx]
            audio_obj = item[cfg["audio"]]
            text = str(item.get(cfg["text"], "")).strip()
            if not text:
                continue
            try:
                arr = np.array(audio_obj["array"], dtype=np.float32)
                sr = audio_obj["sampling_rate"]
                val_samples.append({"audio_array": arr, "sampling_rate": sr, "transcription": cfg["conv"](text)})
            except:
                pass

    del ds; gc.collect()
    print(f"  ✅ ({len(val_samples)} samples cumulés)")

random.shuffle(val_samples)
val_samples = val_samples[:SAMPLE_SIZE]
print(f"\n✅ {len(val_samples)} échantillons prêts pour le baseline")

In [ ]:
# ============================================================
# 🎯 ÉTAPE 3 : Baseline — Test des modèles ASR
# ============================================================
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer
import gc

resultats = {}

# === WHISPER-SMALL ===
print("="*50)
print("🔄 Test de whisper-small...")
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model_ws = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
model_ws.eval()

preds, refs = [], []
for i, s in enumerate(val_samples):
    try:
        inp = processor(s["audio_array"], sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
        with torch.no_grad():
            ids = model_ws.generate(inp)
        text = processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()
        preds.append(text)
        refs.append(s["transcription"].strip().lower())
    except:
        pass
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(val_samples)}")

score = wer(refs, preds)
resultats["whisper-small"] = score
print(f"✅ whisper-small — WER: {score:.2%} ({len(preds)} samples)")
for j in range(min(3, len(preds))):
    print(f"  REF: {refs[j]}")
    print(f"  PRD: {preds[j]}\n")

del model_ws, processor; gc.collect(); torch.cuda.empty_cache()

# === WHISPER-MEDIUM ===
print("="*50)
print("🔄 Test de whisper-medium...")
processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
model_wm = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium").to("cuda")
model_wm.eval()

preds, refs = [], []
for i, s in enumerate(val_samples):
    try:
        inp = processor(s["audio_array"], sampling_rate=16000, return_tensors="pt").input_features.to("cuda")
        with torch.no_grad():
            ids = model_wm.generate(inp)
        text = processor.batch_decode(ids, skip_special_tokens=True)[0].strip().lower()
        preds.append(text)
        refs.append(s["transcription"].strip().lower())
    except:
        pass
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(val_samples)}")

score = wer(refs, preds)
resultats["whisper-medium"] = score
print(f"✅ whisper-medium — WER: {score:.2%} ({len(preds)} samples)")
for j in range(min(3, len(preds))):
    print(f"  REF: {refs[j]}")
    print(f"  PRD: {preds[j]}\n")

del model_wm, processor; gc.collect(); torch.cuda.empty_cache()

# === WAV2VEC2-XLSR (version fine-tunée arabe, plus adaptée) ===
print("="*50)
print("🔄 Test de wav2vec2-xlsr-arabic...")
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic")
model_w2v = Wav2Vec2ForCTC.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic").to("cuda")
model_w2v.eval()

preds, refs = [], []
for i, s in enumerate(val_samples):
    try:
        inp = processor(s["audio_array"], sampling_rate=16000, return_tensors="pt", padding=True).input_values.to("cuda")
        with torch.no_grad():
            logits = model_w2v(inp).logits
        pred_ids = torch.argmax(logits, dim=-1)
        text = processor.batch_decode(pred_ids)[0].strip().lower()
        preds.append(text)
        refs.append(s["transcription"].strip().lower())
    except:
        pass
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(val_samples)}")

score = wer(refs, preds)
resultats["wav2vec2-xlsr-arabic"] = score
print(f"✅ wav2vec2-xlsr-arabic — WER: {score:.2%} ({len(preds)} samples)")
for j in range(min(3, len(preds))):
    print(f"  REF: {refs[j]}")
    print(f"  PRD: {preds[j]}\n")

del model_w2v, processor; gc.collect(); torch.cuda.empty_cache()

In [ ]:
print("\n" + "="*50)
print("📊 RÉSULTATS BASELINE")
print("="*50)

best_model, best_wer = None, float("inf")
for nom, score in resultats.items():
    if score is not None:
        print(f"  {nom:20s} → WER: {score:.2%}")
        if score < best_wer:
            best_wer, best_model = score, nom
    else:
        print(f"  {nom:20s} → ERREUR")

print(f"\n🏆 Meilleur modèle : {best_model} (WER: {best_wer:.2%})")

In [ ]:
# ============================================================
# 🎯 ÉTAPE 4 : Fine-tuning wav2vec2-xlsr → AMAZIGH
# ============================================================
import gc, numpy as np, torch, random
from datasets import load_dataset, Audio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Union
from torch.utils.data import Dataset as TorchDataset
import evaluate

gc.collect()
torch.cuda.empty_cache()

processor = Wav2Vec2Processor.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic")

# === Datasets avec texte ARABE disponible ===
DATASETS_CFG = {
    "Tamazight-NLP/Tamazight-Speech-to-Arabic-Text": {"audio":"audio", "ar":"text"},
    "Tamazight-NLP/Beni-Mellal-Tamazight": {"audio":"Audio", "ar":None, "en":"English"},
    "Tamazight-NLP/TOSD": {"audio":"audio", "ar":None},
}
# Note : Kabyle n'a pas de transcription arabe, on l'exclut du fine-tuning arabe
# Beni-Mellal et TOSD n'ont pas d'arabe non plus -> on les skip

MAX_PER_DATASET = 4000
random.seed(42)

train_data = {"input_values": [], "labels": []}
val_data = {"input_values": [], "labels": []}

# On utilise le dataset qui a le plus d'arabe : Speech-to-Arabic-Text
print("🔄 Tamazight-NLP/Tamazight-Speech-to-Arabic-Text...")
ds = load_dataset("Tamazight-NLP/Tamazight-Speech-to-Arabic-Text")

for split_name, split_ds in ds.items():
    split_ds = split_ds.cast_column("audio", Audio(sampling_rate=16000))
    indices = list(range(len(split_ds)))
    random.shuffle(indices)
    indices = indices[:MAX_PER_DATASET]

    for idx in indices:
        item = split_ds[idx]
        audio_obj = item["audio"]
        text = str(item.get("text", "")).strip()
        if not text:
            continue
        try:
            arr = np.array(audio_obj["array"], dtype=np.float32)
            if len(arr) == 0 or len(arr) > 160000:
                continue
            labels = processor.tokenizer(text, return_tensors="np").input_ids[0].tolist()
            if random.random() < 0.1:
                val_data["input_values"].append(arr)
                val_data["labels"].append(labels)
            else:
                train_data["input_values"].append(arr)
                train_data["labels"].append(labels)
        except:
            pass

del ds; gc.collect()
print(f"  ✅ Train: {len(train_data['input_values'])} | Val: {len(val_data['input_values'])}")

# === DATASET + COLLATOR ===
class ASRDataset(TorchDataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data["input_values"])
    def __getitem__(self, idx):
        return {"input_values": self.data["input_values"][idx], "labels": self.data["labels"][idx]}

train_ds = ASRDataset(train_data)
val_ds = ASRDataset(val_data)

@dataclass
class DataCollatorCTC:
    processor: Wav2Vec2Processor
    def __call__(self, features):
        input_values = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.pad(input_values, padding=True, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=True, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

# === MODÈLE ===
model = Wav2Vec2ForCTC.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic")
model.freeze_feature_encoder()
model = model.to("cuda")

# === TRAINING ===
wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

training_args = TrainingArguments(
    output_dir="/content/wav2vec2-amazigh-ar",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    num_train_epochs=5,
    fp16=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    learning_rate=1e-4,
    warmup_steps=300,
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorCTC(processor=processor),
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print(f"\n🚀 Fine-tuning lancé — Train: {len(train_ds)} | Val: {len(val_ds)}")
trainer.train()

metrics = trainer.evaluate()
print(f"\n📊 WER validation (arabe) : {metrics['eval_wer']:.2%}")

trainer.save_model("/content/wav2vec2-amazigh-final")
processor.save_pretrained("/content/wav2vec2-amazigh-final")
print("✅ Modèle sauvegardé")

In [ ]:
# ============================================================
# 📊 ÉTAPE 5 : Test final — Comparaison avant/après fine-tuning
# ============================================================
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from jiwer import wer
import gc, numpy as np, torch, random
from datasets import load_dataset, Audio

gc.collect()
torch.cuda.empty_cache()

# Charger le modèle fine-tuné
processor = Wav2Vec2Processor.from_pretrained("/content/wav2vec2-amazigh-final")
model_ft = Wav2Vec2ForCTC.from_pretrained("/content/wav2vec2-amazigh-final").to("cuda").eval()

# Charger le modèle original (baseline)
model_base = Wav2Vec2ForCTC.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic").to("cuda").eval()

# Préparer des échantillons de TEST (jamais vus)
print("🔄 Chargement des données de test...")
ds = load_dataset("Tamazight-NLP/Tamazight-Speech-to-Arabic-Text")
test_split = ds["test"].cast_column("audio", Audio(sampling_rate=16000))

random.seed(123)
indices = list(range(len(test_split)))
random.shuffle(indices)
indices = indices[:200]

preds_base, preds_ft, refs = [], [], []

for i, idx in enumerate(indices):
    item = test_split[idx]
    text = str(item["text"]).strip()
    if not text:
        continue
    try:
        arr = np.array(item["audio"]["array"], dtype=np.float32)
        inp = processor(arr, sampling_rate=16000, return_tensors="pt", padding=True).input_values.to("cuda")

        with torch.no_grad():
            logits_base = model_base(inp).logits
            logits_ft = model_ft(inp).logits

        pred_base = processor.batch_decode(torch.argmax(logits_base, dim=-1))[0].strip()
        pred_ft = processor.batch_decode(torch.argmax(logits_ft, dim=-1))[0].strip()

        preds_base.append(pred_base)
        preds_ft.append(pred_ft)
        refs.append(text)
    except:
        pass

    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(indices)}")

# Calcul WER
wer_base = wer(refs, preds_base)
wer_ft = wer(refs, preds_ft)

print(f"\n{'='*50}")
print(f"📊 RÉSULTATS TEST FINAL ({len(refs)} samples)")
print(f"{'='*50}")
print(f"  Baseline (avant)     → WER: {wer_base:.2%}")
print(f"  Fine-tuné (après)    → WER: {wer_ft:.2%}")
print(f"  Amélioration         → {wer_base - wer_ft:.2%}")
print(f"\n📝 Exemples de transcription :")
for j in range(min(5, len(refs))):
    print(f"\n  REF : {refs[j]}")
    print(f"  BASE: {preds_base[j]}")
    print(f"  FT  : {preds_ft[j]}")

del model_base, model_ft; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 📊 ÉTAPE 5 : Vérification de l'overfitting
# ============================================================

print("📊 ANALYSE DE L'OVERFITTING")
print("="*50)
print()
print("Évolution pendant le training :")
print(f"{'Step':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'Val WER':>10}")
print("-"*50)
print(f"{'500':>6} | {'23.881':>12} | {'22.975':>10} | {'99.62%':>10}")
print(f"{'1000':>6} | {'23.094':>12} | {'22.965':>10} | {'98.78%':>10}")
print(f"{'1500':>6} | {'22.269':>12} | {'22.920':>10} | {'98.49%':>10}")

print()
print("Diagnostic :")
train_loss_start, train_loss_end = 23.881, 22.269
val_loss_start, val_loss_end = 22.975, 22.920

print(f"  Train loss : {train_loss_start:.3f} → {train_loss_end:.3f} (↓ {train_loss_start-train_loss_end:.3f})")
print(f"  Val loss   : {val_loss_start:.3f} → {val_loss_end:.3f} (↓ {val_loss_start-val_loss_end:.3f})")

if train_loss_end < val_loss_end - 1:
    print("\n  ⚠️ OVERFITTING détecté : train loss << val loss")
else:
    print("\n  ✅ PAS D'OVERFITTING : train loss et val loss évoluent de manière similaire")



In [ ]:
# ============================================================
# 📊 ÉTAPE 5 : Vérification de l'overfitting
# ============================================================

print("📊 ANALYSE DE L'OVERFITTING")
print("="*50)
print()
print("Évolution pendant le training :")
print(f"{'Step':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'Val WER':>10}")
print("-"*50)
print(f"{'500':>6} | {'23.881':>12} | {'22.975':>10} | {'99.62%':>10}")
print(f"{'1000':>6} | {'23.094':>12} | {'22.965':>10} | {'98.78%':>10}")
print(f"{'1500':>6} | {'22.269':>12} | {'22.920':>10} | {'98.49%':>10}")

print()
print("Diagnostic :")
train_loss_start, train_loss_end = 23.881, 22.269
val_loss_start, val_loss_end = 22.975, 22.920

print(f"  Train loss : {train_loss_start:.3f} → {train_loss_end:.3f} (↓ {train_loss_start-train_loss_end:.3f})")
print(f"  Val loss   : {val_loss_start:.3f} → {val_loss_end:.3f} (↓ {val_loss_start-val_loss_end:.3f})")

if train_loss_end < val_loss_end - 1:
    print("\n  ⚠️ OVERFITTING détecté : train loss << val loss")
else:
    print("\n  ✅ PAS D'OVERFITTING : train loss et val loss évoluent de manière similaire")

